# eval_generative.ipynb

Evaluation notebook for the trained conditional VAE (`train_cvae.py`).

**Sections:**
1. Load model + data
2. Grid of generated samples at fixed z_lens, varying θ_E
3. Power spectrum: generated vs. real
4. Reconstruction quality: encode → decode → residual
5. t-SNE of latent space colored by lensed/non-lensed label

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

# ── Config ─────────────────────────────────────────────────────────────────
DATA_DIR    = 'output/v2'
WEIGHTS     = 'output/v2/cvae_weights_best.pt'
LATENT_DIM  = 32
COND_DIM    = 4
IMAGE_SIZE  = 128

# Check that weights exist
if not os.path.exists(WEIGHTS):
    raise FileNotFoundError(
        f'{WEIGHTS} not found. Run `python train_cvae.py` first.'
    )
print(f'Weights found: {WEIGHTS}')

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
import torch

# Import the CVAE architecture from train_cvae.py
import importlib, sys
sys.path.insert(0, '.')
import train_cvae

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

model = train_cvae.build_model(LATENT_DIM, COND_DIM, IMAGE_SIZE).to(device)
model.load_state_dict(torch.load(WEIGHTS, map_location=device))
model.eval()
print('Model loaded.')

In [ ]:
# ── Load and preprocess real data ──────────────────────────────────────────
images_raw, cond_np = train_cvae.load_data(DATA_DIR)

theta_Es = np.load(f'{DATA_DIR}/theta_Es.npy')
z_lens   = np.load(f'{DATA_DIR}/z_lens.npy')
z_source = np.load(f'{DATA_DIR}/z_source.npy')
lensed   = np.load(f'{DATA_DIR}/lensed.npy')

X_tensor = torch.tensor(images_raw).to(device)
C_tensor = torch.tensor(cond_np).to(device)
N = len(images_raw)
print(f'Loaded {N} images.  Lensed: {int(lensed.sum())}  Non-lensed: {int((lensed==0).sum())}')

def to_display(t):
    """Convert normalized [-1,1] image tensor to display array."""
    arr = t.detach().cpu().numpy()
    if arr.ndim == 4:
        arr = arr[:, 0]     # (N, H, W)
    elif arr.ndim == 3:
        arr = arr[0]        # (H, W)
    return arr

In [ ]:
# ── Training loss curve ────────────────────────────────────────────────────
log_path = f'{DATA_DIR}/cvae_train_log.npy'
if os.path.exists(log_path):
    log = np.load(log_path)   # (epochs, 3): [epoch, train_loss, val_loss]
    fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=100)
    ax.plot(log[:, 0], log[:, 1], label='Train loss')
    ax.plot(log[:, 0], log[:, 2], label='Val loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ELBO loss')
    ax.set_title('cVAE training curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{DATA_DIR}/eval_loss_curve.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Final train={log[-1,1]:.4f}  val={log[-1,2]:.4f}')
else:
    print(f'Log not found at {log_path}')

In [ ]:
# ── Section 2: Generated samples at fixed z_lens, varying theta_E ──────────
# Shows how the model captures the Einstein radius -> arc size relationship.

theta_E_vals = np.linspace(0.4, 1.4, 8)    # 8 values across the SLACS range
z_lens_fixed  = 0.3
z_source_fixed = 1.5
n_samples_each = 4

fig, axes = plt.subplots(n_samples_each, len(theta_E_vals),
                          figsize=(3*len(theta_E_vals), 3*n_samples_each), dpi=100)

with torch.no_grad():
    for col, tE in enumerate(theta_E_vals):
        # Build condition vector for n_samples_each identical conditions
        cond_fixed = torch.tensor(
            np.tile(
                [[tE/2.0, z_lens_fixed/0.9, z_source_fixed/3.0, 1.0]],
                (n_samples_each, 1)
            ),
            dtype=torch.float32, device=device
        )
        samples = model.sample(cond_fixed, n=n_samples_each, device=device)
        imgs = to_display(samples)

        for row in range(n_samples_each):
            ax = axes[row, col]
            ax.imshow(imgs[row], cmap='gray', origin='lower',
                      vmin=imgs[row].min(), vmax=imgs[row].max())
            ax.axis('off')
            if row == 0:
                ax.set_title(f'θ_E={tE:.2f}"', fontsize=9)

axes[0, 0].set_ylabel(f'z_l={z_lens_fixed}\nz_s={z_source_fixed}', fontsize=9)
plt.suptitle('cVAE: generated lensed images (fixed z_l, varying θ_E)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/eval_generated_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved -> {DATA_DIR}/eval_generated_grid.png')

In [ ]:
# ── Section 3: Power spectrum — generated vs. real ─────────────────────────
# Azimuthally averaged 2D power spectrum (P(k) curve).

def azimuthal_power_spectrum(img):
    """Return (k_bins, P_k) from a 2D image."""
    h, w  = img.shape
    fft2  = np.fft.fftshift(np.fft.fft2(img))
    power = np.abs(fft2)**2

    ky = np.fft.fftshift(np.fft.fftfreq(h))
    kx = np.fft.fftshift(np.fft.fftfreq(w))
    KX, KY = np.meshgrid(kx, ky)
    K = np.sqrt(KX**2 + KY**2)

    k_max = 0.5
    bins  = np.linspace(0, k_max, 40)
    Pk    = np.zeros(len(bins) - 1)
    for i in range(len(bins) - 1):
        mask  = (K >= bins[i]) & (K < bins[i+1])
        if mask.sum() > 0:
            Pk[i] = power[mask].mean()
    k_centers = 0.5 * (bins[:-1] + bins[1:])
    return k_centers, Pk


# Real images: random subset of lensed
lensed_idx = np.where(lensed == 1)[0]
n_ps = min(100, len(lensed_idx))
real_subset = images_raw[lensed_idx[:n_ps], 0]   # (n_ps, 128, 128)

# Generated images: random samples with lensed condition
cond_gen = torch.tensor(
    np.column_stack([
        np.random.uniform(0.5, 1.5, n_ps) / 2.0,
        np.random.uniform(0.1, 0.7, n_ps) / 0.9,
        np.random.uniform(0.8, 2.5, n_ps) / 3.0,
        np.ones(n_ps)
    ]).astype(np.float32),
    device=device
)
with torch.no_grad():
    gen_imgs = model.sample(cond_gen, n=n_ps, device=device)
gen_subset = to_display(gen_imgs)   # (n_ps, 128, 128)

# Average power spectra
k_real, Pk_real = zip(*[azimuthal_power_spectrum(img) for img in real_subset])
k_gen,  Pk_gen  = zip(*[azimuthal_power_spectrum(img) for img in gen_subset])

k_real = np.array(k_real)[0]
Pk_real_mean = np.array(Pk_real).mean(axis=0)
Pk_gen_mean  = np.array(Pk_gen).mean(axis=0)
Pk_real_std  = np.array(Pk_real).std(axis=0)
Pk_gen_std   = np.array(Pk_gen).std(axis=0)

fig, ax = plt.subplots(1, 1, figsize=(8, 5), dpi=100)
ax.semilogy(k_real, Pk_real_mean, 'b-', label='Real (lensed)', lw=2)
ax.fill_between(k_real,
                np.maximum(Pk_real_mean - Pk_real_std, 1e-10),
                Pk_real_mean + Pk_real_std, alpha=0.3, color='blue')
ax.semilogy(k_real, Pk_gen_mean,  'r--', label='Generated (cVAE)', lw=2)
ax.fill_between(k_real,
                np.maximum(Pk_gen_mean - Pk_gen_std, 1e-10),
                Pk_gen_mean + Pk_gen_std, alpha=0.3, color='red')
ax.set_xlabel('Spatial frequency k (cycles/pix)')
ax.set_ylabel('Power P(k)')
ax.set_title(f'Azimuthally averaged power spectrum (n={n_ps})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/eval_power_spectrum.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved -> {DATA_DIR}/eval_power_spectrum.png')

In [ ]:
# ── Section 4: Reconstruction quality ──────────────────────────────────────
# Encode real images -> z -> decode -> compare

n_show = 8
idx_show = lensed_idx[:n_show]

X_show = X_tensor[idx_show]
C_show = C_tensor[idx_show]

with torch.no_grad():
    recon, mu, logvar = model(X_show, C_show)

X_np    = to_display(X_show)
recon_np = to_display(recon)
resid_np = X_np - recon_np

fig, axes = plt.subplots(3, n_show, figsize=(3*n_show, 10), dpi=100)
for col in range(n_show):
    def show(ax, img, title, cmap='gray'):
        vmin, vmax = np.percentile(img, [1, 99.5])
        ax.imshow(img, cmap=cmap, origin='lower', vmin=vmin, vmax=vmax)
        ax.set_title(title, fontsize=7)
        ax.axis('off')

    tE = theta_Es[idx_show[col]]
    show(axes[0, col], X_np[col],    f'Real θ_E={tE:.2f}"')
    show(axes[1, col], recon_np[col], 'Recon')
    show(axes[2, col], resid_np[col], 'Residual', cmap='RdBu_r')

axes[0, 0].set_ylabel('Real', fontsize=10)
axes[1, 0].set_ylabel('Reconstructed', fontsize=10)
axes[2, 0].set_ylabel('Residual', fontsize=10)

mse = float(((X_np - recon_np)**2).mean())
plt.suptitle(f'cVAE reconstruction  MSE={mse:.5f}', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/eval_reconstruction.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved -> {DATA_DIR}/eval_reconstruction.png')
print(f'Mean reconstruction MSE: {mse:.6f}')

In [ ]:
# ── Section 5: t-SNE of latent space ──────────────────────────────────────
# Encode all images -> extract mu -> project to 2D with t-SNE
# Color by lensed/non-lensed label

from sklearn.manifold import TSNE

BATCH = 256
mus = []
with torch.no_grad():
    for i in range(0, N, BATCH):
        xb = X_tensor[i:i+BATCH]
        cb = C_tensor[i:i+BATCH]
        mu_b, _ = model.encode(xb, cb)
        mus.append(mu_b.cpu().numpy())
mus = np.concatenate(mus, axis=0)   # (N, LATENT_DIM)
print(f'Latent embeddings: {mus.shape}')

# Subsample to keep t-SNE fast
n_tsne = min(2000, N)
idx_tsne = np.random.choice(N, n_tsne, replace=False)
mus_sub = mus[idx_tsne]
labels_sub = lensed[idx_tsne]
tE_sub = theta_Es[idx_tsne]

print(f'Running t-SNE on {n_tsne} samples...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
Z2 = tsne.fit_transform(mus_sub)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100)

# Left: color by lensed/non-lensed
for label_val, label_name, color in [(0, 'Non-lensed', 'royalblue'),
                                       (1, 'Lensed', 'tomato')]:
    mask = labels_sub == label_val
    axes[0].scatter(Z2[mask, 0], Z2[mask, 1], s=5, alpha=0.5,
                    c=color, label=label_name)
axes[0].set_title('t-SNE latent space: lensed vs. non-lensed')
axes[0].legend(markerscale=3)
axes[0].axis('off')

# Right: color by theta_E (lensed only)
lensed_mask = labels_sub == 1
sc = axes[1].scatter(Z2[lensed_mask, 0], Z2[lensed_mask, 1],
                     s=8, alpha=0.7,
                     c=tE_sub[lensed_mask], cmap='plasma',
                     vmin=0.5, vmax=1.5)
axes[1].scatter(Z2[~lensed_mask, 0], Z2[~lensed_mask, 1],
                s=4, alpha=0.2, c='gray')
plt.colorbar(sc, ax=axes[1], label='θ_E (arcsec)')
axes[1].set_title('t-SNE latent space: θ_E gradient (lensed only)')
axes[1].axis('off')

plt.suptitle('cVAE latent space structure', fontsize=13)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/eval_tsne.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved -> {DATA_DIR}/eval_tsne.png')

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────
print('=== Evaluation complete ===')
outputs = [
    'eval_loss_curve.png',
    'eval_generated_grid.png',
    'eval_power_spectrum.png',
    'eval_reconstruction.png',
    'eval_tsne.png',
]
for f in outputs:
    p = os.path.join(DATA_DIR, f)
    exists = '✓' if os.path.exists(p) else '✗ missing'
    print(f'  {exists}  {p}')